# VoiceMorph — export FreeVC to ONNX

Produces the three files VoiceMorph needs: `content.onnx`, `speaker.onnx`,
`decoder.onnx`.

**Runtime → Change runtime type → T4 GPU** before you start. CPU works but is slow.

Run the cells in order. Cells 6 and 9 are verification steps — they compare the
ONNX output against the original PyTorch and print a similarity number. If those
numbers are wrong, everything downstream is wrong, and you want to know that here
rather than by listening to noise later.


## 1. Install and preflight

This cell installs everything the whole notebook needs, then imports every
single module to prove it. Nothing further down can fail on a missing package:
if something is absent it stops here, loudly, before you spend twenty minutes
downloading checkpoints.

It also defines `export_onnx()`, the helper all three export cells use.


In [ ]:
import importlib, subprocess, sys, inspect

# module name -> pip name
REQUIRED = {
    "onnx":            "onnx",
    "onnxruntime":     "onnxruntime",
    "onnxscript":      "onnxscript",      # torch's newer exporter imports this
    "librosa":         "librosa>=0.10",
    "soundfile":       "soundfile",
    "huggingface_hub": "huggingface_hub",
    "webrtcvad":       "webrtcvad-wheels",  # FreeVC needs it; prebuilt, no compiler
    "scipy":           "scipy",
    "matplotlib":      "matplotlib",
    "numpy":           "numpy",
    "torch":           "torch",
    "torchaudio":      "torchaudio",
}

missing = []
for module, package in REQUIRED.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(package)

if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    importlib.invalidate_caches()

still_broken = []
for module in REQUIRED:
    try:
        importlib.import_module(module)
    except ImportError as e:
        still_broken.append(f"{module}: {e}")

if still_broken:
    raise SystemExit("STOP - could not import:\n  " + "\n  ".join(still_broken))

import torch, onnx
print("torch", torch.__version__, "| onnx", onnx.__version__,
      "| cuda", torch.cuda.is_available())
print("all", len(REQUIRED), "dependencies present")


# --------------------------------------------------------------------------
def export_onnx(module, args, path, input_names, output_names,
                dynamic_axes, opset=17):
    """Exports and immediately validates.

    torch 2.5 added a dynamo-based exporter that is on by default in some
    builds. It does not cope with the dynamic control flow in VITS, so the
    legacy tracer is forced where the argument exists.
    """
    kwargs = dict(input_names=input_names, output_names=output_names,
                  dynamic_axes=dynamic_axes, opset_version=opset,
                  do_constant_folding=True)

    if "dynamo" in inspect.signature(torch.onnx.export).parameters:
        kwargs["dynamo"] = False

    module.eval()
    with torch.no_grad():
        torch.onnx.export(module, args, path, **kwargs)

    onnx.checker.check_model(onnx.load(path))

    import os
    print(f"{path}  ({os.path.getsize(path) / 1e6:.0f} MB)  - graph validates")


## 2. Get the FreeVC source

In [ ]:
!git clone -q https://github.com/OlaWod/FreeVC.git
%cd /content/FreeVC
sys.path.insert(0, "/content/FreeVC")
!ls

## 3. Patch FreeVC for modern librosa

FreeVC was written against librosa 0.8, which accepted positional arguments.
librosa 0.10 made them keyword-only, so the speaker encoder throws a TypeError
the moment you touch it.

Pinning old librosa is the other way out, but it drags in a numba and numpy
combination that fights with the rest of Colab. Patching two call sites is the
smaller change.


In [ ]:
import re
from pathlib import Path

target = Path("speaker_encoder/audio.py")
source = target.read_text()
before = source

source = re.sub(r"melspectrogram\(\s*wav\s*,\s*sampling_rate\s*,",
                "melspectrogram(\n        y=wav,\n        sr=sampling_rate,",
                source)

source = re.sub(r"librosa\.resample\(\s*wav\s*,\s*source_sr\s*,\s*sampling_rate\s*\)",
                "librosa.resample(wav, orig_sr=source_sr, target_sr=sampling_rate)",
                source)

if source != before:
    target.write_text(source)
    print("patched speaker_encoder/audio.py")
else:
    print("nothing to patch - either already fixed, or the code has moved.")
    print("If cell 5 throws a librosa TypeError, paste the traceback back.")

# Quick proof the import chain now works end to end.
import importlib, sys
sys.path.insert(0, "/content/FreeVC")
from speaker_encoder.voice_encoder import SpeakerEncoder
print("speaker encoder imports cleanly")

## 4. Fetch the checkpoints

Three weights files are needed:

| What | Goes to | Size |
|---|---|---|
| FreeVC generator | `checkpoints/freevc.pth` | ~470 MB |
| Speaker encoder | `speaker_encoder/ckpt/pretrained_bak_5805000.pt` | ~17 MB |
| WavLM-Large (content encoder) | `wavlm/WavLM-Large.pt` | ~1.2 GB |

The first two live in the FreeVC Space. WavLM-Large does not — Microsoft
publishes it through the unilm repo behind expiring links, so everyone relies
on community mirrors. This cell tries several and checks the SHA-256 of what
it gets, because a mirror that quietly serves the wrong file would show up much
later as a voice that is merely bad rather than as an error.


In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download
from pathlib import Path
import shutil, hashlib

# --- freevc.pth and the speaker encoder, from the FreeVC Space --------------
SPACE = "OlaWod/FreeVC"
try:
    space_files = list_repo_files(SPACE, repo_type="space")
except Exception as e:
    print("Could not list the Space:", e)
    space_files = []

for dest, needle in [("checkpoints/freevc.pth", "freevc.pth"),
                     ("speaker_encoder/ckpt/pretrained_bak_5805000.pt", "pretrained_bak_5805000.pt")]:
    if Path(dest).exists():
        print("already here:", dest); continue

    match = next((f for f in space_files if needle in f), None)
    if match is None:
        print("NOT FOUND in the Space:", needle); continue

    print("downloading", match)
    src = hf_hub_download(SPACE, match, repo_type="space")
    Path(dest).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dest)

# --- WavLM-Large, from whichever mirror is still alive ----------------------
WAVLM_SHA256 = "6fb4b3c3e6aa567f0a997b30855859cb81528ee8078802af439f7b2da0bf100f"
WAVLM_SIZE   = 1261965425

MIRRORS = [
    ("MrDragonFox/LLaSE-G1",      "WavLM-Large.pt", "model"),
    ("MercuryLeafer/NeuCoSVC-2",  "WavLM-Large.pt", "space"),
]

def sha256_of(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

wavlm = Path("wavlm/WavLM-Large.pt")
wavlm.parent.mkdir(parents=True, exist_ok=True)

if wavlm.exists() and wavlm.stat().st_size == WAVLM_SIZE:
    print("WavLM already here")
else:
    for repo, filename, kind in MIRRORS:
        try:
            print(f"trying {repo} ({kind}) - this is 1.2 GB, give it a few minutes")
            src = hf_hub_download(repo, filename, repo_type=kind)

            size = Path(src).stat().st_size
            if size != WAVLM_SIZE:
                print(f"  wrong size: got {size}, expected {WAVLM_SIZE}. Skipping.")
                continue

            print("  verifying checksum")
            digest = sha256_of(src)
            if digest != WAVLM_SHA256:
                print(f"  checksum mismatch: {digest}. Skipping this mirror.")
                continue

            shutil.copy(src, wavlm)
            print("  verified and installed")
            break
        except Exception as e:
            print("  failed:", type(e).__name__, e)
    else:
        print("\nEvery mirror failed. Download WavLM-Large.pt by hand from")
        print("https://github.com/microsoft/unilm/tree/master/wavlm and upload it")
        print("to wavlm/WavLM-Large.pt using the Colab file browser on the left.")

# --- report -----------------------------------------------------------------
print()
for p in ["checkpoints/freevc.pth",
          "speaker_encoder/ckpt/pretrained_bak_5805000.pt",
          "wavlm/WavLM-Large.pt"]:
    f = Path(p)
    print(("OK   " if f.exists() else "MISS "), p,
          f"({f.stat().st_size / 1e6:.0f} MB)" if f.exists() else "")


## 5. Load the PyTorch models

In [ ]:
import torch, utils
from models import SynthesizerTrn
from speaker_encoder.voice_encoder import SpeakerEncoder
from wavlm import WavLM, WavLMConfig

device = "cpu"   # export on CPU: tracing is more predictable and this is not slow

hps = utils.get_hparams_from_file("configs/freevc.json")

net_g = SynthesizerTrn(hps.data.filter_length // 2 + 1,
                       hps.train.segment_size // hps.data.hop_length,
                       **hps.model).to(device)
net_g.eval()
utils.load_checkpoint("checkpoints/freevc.pth", net_g, None, True)

ckpt = torch.load("wavlm/WavLM-Large.pt", map_location="cpu")
cmodel = WavLM(WavLMConfig(ckpt["cfg"]))
cmodel.load_state_dict(ckpt["model"])
cmodel.eval().to(device)

smodel = SpeakerEncoder("speaker_encoder/ckpt/pretrained_bak_5805000.pt", device="cpu")

print("loaded")

## 6. Export the content encoder

WavLM turns 16 kHz speech into frames that describe *what was said* without the
speaker's identity. FreeVC feeds the decoder a `[1, 1024, T]` tensor, so the
transpose goes inside the graph — the plugin passes the content output straight
through to the decoder and does no reshaping of its own.


In [ ]:
import torch.nn as nn

class ContentEncoder(nn.Module):
    def __init__(self, wavlm):
        super().__init__()
        self.wavlm = wavlm

    def forward(self, audio):                            # [1, samples] @ 16 kHz
        feats = self.wavlm.extract_features(audio)[0]    # [1, T, 1024]
        return feats.transpose(1, 2)                     # [1, 1024, T]

export_onnx(ContentEncoder(cmodel),
            (torch.randn(1, 16000 * 3),),
            "content.onnx",
            input_names=["audio"], output_names=["features"],
            dynamic_axes={"audio": {1: "samples"}, "features": {2: "frames"}})


## 7. Export the speaker encoder

Two problems here, both worth understanding.

**FreeVC computes the mel spectrogram in numpy, outside the network.** A naive
export therefore produces a model that wants mel frames, and the plugin only
has audio. So the mel goes into the graph. The filterbank is lifted straight
out of librosa as a constant, which removes any chance of a slaney-versus-htk
mismatch — a bug that yields a plausible-looking embedding describing the wrong
voice, with no error anywhere.

**`torch.stft` cannot be exported.** It returns complex numbers and the ONNX
exporter refuses them outright. The fix is to express the STFT as what it
actually is: a bank of fixed filters. Each DFT frequency becomes two
convolution kernels, cosine and sine, windowed by the same Hann. A strided
`conv1d` against them is arithmetically identical to a framed `rfft` — verified
to 3e-14 against numpy — and ONNX sees nothing but an ordinary Conv.


In [ ]:
import math, librosa, numpy as np, torch.nn as nn
import torch.nn.functional as F

SR, N_FFT, HOP, N_MELS = 16000, 400, 160, 40

mel_fb = librosa.filters.mel(sr=SR, n_fft=N_FFT, n_mels=N_MELS)   # [40, 201]

class SpeakerEmbedder(nn.Module):
    def __init__(self, se, fb):
        super().__init__()
        self.lstm   = se.lstm
        self.linear = se.linear

        # STFT as a convolution: cos and sin kernels, Hann-windowed.
        n     = torch.arange(N_FFT, dtype=torch.float32)
        k     = torch.arange(N_FFT // 2 + 1, dtype=torch.float32).unsqueeze(1)
        angle = 2.0 * math.pi * k * n / N_FFT
        window = torch.hann_window(N_FFT)                 # periodic, like librosa

        self.register_buffer("kernel_real", ( torch.cos(angle) * window).unsqueeze(1))
        self.register_buffer("kernel_imag", (-torch.sin(angle) * window).unsqueeze(1))
        self.register_buffer("fb", torch.from_numpy(fb).float())

    def forward(self, audio):                          # [1, samples] @ 16 kHz
        x = audio.unsqueeze(1)                         # [1, 1, samples]
        x = F.pad(x, (N_FFT // 2, N_FFT // 2), mode="constant")   # center=True

        real = F.conv1d(x, self.kernel_real, stride=HOP)   # [1, 201, T]
        imag = F.conv1d(x, self.kernel_imag, stride=HOP)
        power = real * real + imag * imag

        mel = torch.matmul(self.fb, power)             # [1, 40, T]
        mel = mel.transpose(1, 2)                      # [1, T, 40]

        _, (hidden, _) = self.lstm(mel)
        raw = torch.relu(self.linear(hidden[-1]))
        return raw / torch.norm(raw, dim=1, keepdim=True)          # [1, 256]

export_onnx(SpeakerEmbedder(smodel, mel_fb),
            (torch.randn(1, 16000 * 5),),
            "speaker.onnx",
            input_names=["audio"], output_names=["embedding"],
            dynamic_axes={"audio": {1: "samples"}})


## 8. Check the speaker embedding — do not skip this

Compares the exported graph against FreeVC's own `embed_utterance` on a real
recording. Cosine similarity is the number that matters.

- **above 0.99** — the export is faithful, carry on
- **0.90 to 0.99** — the mel matches but the whole-utterance LSTM drifts from the
  reference's partial averaging. Usable. Feeding shorter references (3–6 s)
  usually pushes it back up
- **below 0.90** — something is genuinely wrong. Send me the number and the mel
  shapes printed below


In [ ]:
import onnxruntime as ort, numpy as np, librosa

wav_path = "p225_001.wav"
if not Path(wav_path).exists():
    hf = hf_hub_download("OlaWod/FreeVC", "p225_001.wav", repo_type="space")
    shutil.copy(hf, wav_path)

wav, _ = librosa.load(wav_path, sr=SR)
print("reference length: %.1f s" % (len(wav) / SR))

reference = smodel.embed_utterance(wav)                     # numpy [256]

sess = ort.InferenceSession("speaker.onnx", providers=["CPUExecutionProvider"])
exported = sess.run(["embedding"], {"audio": wav[None, :].astype(np.float32)})[0][0]

cos = float(np.dot(reference, exported) /
            (np.linalg.norm(reference) * np.linalg.norm(exported)))

print("cosine similarity: %.4f" % cos)
print("verdict:", "good" if cos > 0.99 else ("usable" if cos > 0.90 else "BROKEN"))

## 9. Export the decoder

One subtlety: FreeVC's `infer()` does `g = g.unsqueeze(-1)` internally, so it
wants the speaker vector as a flat `[1, 256]` and adds the trailing axis
itself. Handing it `[1, 256, 1]` produces a 4-D tensor and conv1d refuses it.


In [ ]:
import torch.nn as nn

class Decoder(nn.Module):
    def __init__(self, g):
        super().__init__()
        self.net_g = g

    def forward(self, features, speaker):     # [1,1024,T] and [1,256]
        return self.net_g.infer(features, g=speaker)

export_onnx(Decoder(net_g),
            (torch.randn(1, 1024, 150), torch.randn(1, 256)),
            "decoder.onnx",
            input_names=["features", "speaker"], output_names=["audio"],
            dynamic_axes={"features": {2: "frames"}, "audio": {2: "samples"}})


## 10. End-to-end test in pure ONNX

No PyTorch in this cell. If the audio player below sounds like the target voice
saying the source words, all three graphs are correct and the plugin will work.


In [ ]:
from IPython.display import Audio, display
import soundfile as sf

src_path, tgt_path = "p225_001.wav", "p226_002.wav"
if not Path(tgt_path).exists():
    shutil.copy(hf_hub_download("OlaWod/FreeVC", tgt_path, repo_type="space"), tgt_path)

src, _ = librosa.load(src_path, sr=SR)
tgt, _ = librosa.load(tgt_path, sr=SR)

s_content = ort.InferenceSession("content.onnx", providers=["CPUExecutionProvider"])
s_speaker = ort.InferenceSession("speaker.onnx", providers=["CPUExecutionProvider"])
s_decoder = ort.InferenceSession("decoder.onnx", providers=["CPUExecutionProvider"])

feats = s_content.run(["features"],  {"audio": src[None, :].astype(np.float32)})[0]
emb   = s_speaker.run(["embedding"], {"audio": tgt[None, :].astype(np.float32)})[0]

out = s_decoder.run(["audio"], {"features": feats, "speaker": emb})[0]

audio = np.asarray(out).reshape(-1)
sf.write("converted.wav", audio, SR)

print("source:");    display(Audio(src, rate=SR))
print("target:");    display(Audio(tgt, rate=SR))
print("converted:"); display(Audio(audio, rate=SR))

## 11. Download

Extract the zip into one folder, then in VoiceMorph click **Load models folder**
and point at it. The three filenames must stay exactly as they are.


In [ ]:
!mkdir -p voicemorph_models
!cp content.onnx speaker.onnx decoder.onnx voicemorph_models/
!cd voicemorph_models && zip -q -r ../voicemorph_models.zip .
!ls -lh voicemorph_models.zip

from google.colab import files
files.download("voicemorph_models.zip")